# Kinarm Multi-Trial DDM Analysis

This notebook processes Kinarm data from multiple trials across STAT and INTER blocks and fits Drift Diffusion Models (DDMs) to the reaction time distributions.

## Data Structure Expected

```
Data Directory/
├── CMT003_01_STAT.kinarm/
│   ├── Trial1.TP13.C1.csv
│   ├── Trial2.TP14.C2.csv
│   └── ...
├── CMT003_02_STAT.kinarm/
├── CMT003_05_INTER.kinarm/
│   ├── Trial1.TP5.C1.csv
│   ├── Trial2.TP6.C2.csv
│   └── ...
└── ...
```

## Block Types

### STAT Blocks (Stationary)
- **TP13**: Right hand
- **TP14**: Left hand

### INTER Blocks (Interception)
- **TP5**: Right hand at 75°/s
- **TP6**: Left hand at 75°/s
- **TP9**: Right hand at 150°/s
- **TP10**: Left hand at 150°/s

In [1]:
# Install required packages (uncomment if needed)
# !pip install pandas numpy matplotlib pyddm --break-system-packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

try:
    from ddm import Model, Sample, Fittable
    from ddm.functions import fit_adjust_model
    from ddm.models import DriftConstant, NoiseConstant, BoundConstant, OverlayNonDecision, ICPointSourceCenter
    PYDDM_AVAILABLE = True
    print("✓ PyDDM loaded successfully")
except ImportError:
    print("⚠ PyDDM not installed. Install with: pip install pyddm --break-system-packages")
    PYDDM_AVAILABLE = False

## 1. Configure Data Path

In [ ]:
# UPDATE THIS PATH to your data directory
DATA_PATH = r"C:\Users\Rishv\Downloads\CMT003_PRE\CMT003_PRE"

# Subject pattern to match (usually the subject ID)
SUBJECT_PATTERN = 'CMT003'

# Velocity threshold for movement onset detection (m/s)
VELOCITY_THRESHOLD = 0.05

# Sampling rate (Hz) - adjust if different
SAMPLING_RATE = 100  # 100 Hz = 0.01s per frame

## 2. Data Loading Functions

In [4]:
def extract_reaction_time(trial_df, velocity_threshold=0.05, sampling_rate=100):
    """
    Extract reaction time from a single trial.
    RT = time from target appearance to movement onset
    """
    if 'Right_HandVelX' in trial_df.columns and 'Right_HandVelY' in trial_df.columns:
        # Calculate velocity magnitude
        vel_mag = np.sqrt(trial_df['Right_HandVelX']**2 + trial_df['Right_HandVelY']**2)
        
        # Find movement onset
        movement_onset = vel_mag[vel_mag > velocity_threshold].index
        
        if len(movement_onset) > 0:
            onset_frame = movement_onset[0]
            rt_seconds = onset_frame / sampling_rate
            return rt_seconds
    
    return None


def parse_trial_filename(filename):
    """Parse trial filename to extract metadata."""
    parts = filename.replace('.csv', '').split('.')
    info = {}
    
    for part in parts:
        if part.startswith('Trial'):
            info['trial_num'] = int(part.replace('Trial', ''))
        elif part.startswith('TP'):
            info['tp'] = int(part.replace('TP', ''))
        elif part.startswith('C'):
            info['condition'] = int(part.replace('C', ''))
    
    return info


def parse_block_folder(folder_name):
    """Parse block folder name to extract metadata."""
    parts = folder_name.replace('.kinarm', '').split('_')
    
    return {
        'subject': parts[0] if len(parts) > 0 else None,
        'block_num': int(parts[1]) if len(parts) > 1 else None,
        'block_type': parts[2] if len(parts) > 2 else None
    }


def get_hand_and_speed(tp, block_type):
    """Determine hand and speed from TP code and block type."""
    if block_type == 'STAT':
        if tp == 13:
            return 'Right', 0
        elif tp == 14:
            return 'Left', 0
    elif block_type == 'INTER':
        if tp == 5:
            return 'Right', 75
        elif tp == 6:
            return 'Left', 75
        elif tp == 9:
            return 'Right', 150
        elif tp == 10:
            return 'Left', 150
    
    return None, None

print("✓ Functions defined")

✓ Functions defined


## 3. Load All Trials

In [3]:
def load_all_trials(base_path, subject_pattern='CMT003'):
    """Load all trials from all blocks."""
    base_path = Path(base_path)
    block_folders = sorted(base_path.glob(f'{subject_pattern}*'))
    
    print(f"Found {len(block_folders)} block folders")
    
    all_trials = []
    
    for block_folder in block_folders:
        if not block_folder.is_dir():
            continue
        
        block_info = parse_block_folder(block_folder.name)
        csv_files = sorted(block_folder.glob('*.csv'))
        
        print(f"\nProcessing {block_folder.name}: {len(csv_files)} trials")
        
        for csv_file in csv_files:
            try:
                trial_df = pd.read_csv(csv_file)
                rt = extract_reaction_time(trial_df, VELOCITY_THRESHOLD, SAMPLING_RATE)
                
                if rt is not None:
                    trial_info = parse_trial_filename(csv_file.name)
                    hand, speed = get_hand_and_speed(trial_info['tp'], block_info['block_type'])
                    
                    trial_record = {
                        **block_info,
                        **trial_info,
                        'hand': hand,
                        'speed': speed,
                        'rt_seconds': rt,
                        'rt_ms': rt * 1000,
                        'filename': csv_file.name
                    }
                    
                    all_trials.append(trial_record)
            
            except Exception as e:
                print(f"  ⚠ Could not process {csv_file.name}: {e}")
    
    df = pd.DataFrame(all_trials)
    
    # Create condition label
    df['condition'] = df.apply(
        lambda row: f"{row['block_type']}_{row['hand']}_{row['speed']}deg" 
        if pd.notna(row['hand']) else None,
        axis=1
    )
    
    print(f"\n{'='*60}")
    print(f"Total trials loaded: {len(df)}")
    print(f"Block types: {df['block_type'].value_counts().to_dict()}")
    print(f"{'='*60}")
    
    return df


# Load the data
trial_data = load_all_trials(DATA_PATH, SUBJECT_PATTERN)

NameError: name 'DATA_PATH' is not defined

## 4. Exploratory Data Analysis

In [ ]:
# Display first few rows
print("Sample of loaded data:")
display(trial_data.head(10))

# Summary statistics by condition
print("\nSummary by condition:")
summary = trial_data.groupby(['block_type', 'hand', 'speed'])['rt_ms'].agg([
    ('count', 'count'),
    ('mean', 'mean'),
    ('std', 'std'),
    ('median', 'median'),
    ('min', 'min'),
    ('max', 'max')
]).round(2)

display(summary)

In [ ]:
# Save trial data to CSV
output_file = 'kinarm_trial_summary.csv'
trial_data.to_csv(output_file, index=False)
print(f"✓ Trial data saved to: {output_file}")

## 5. Visualize RT Distributions

In [ ]:
# STAT blocks
stat_data = trial_data[trial_data['block_type'] == 'STAT']

if len(stat_data) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Right vs Left
    stat_right = stat_data[stat_data['hand'] == 'Right']['rt_ms']
    stat_left = stat_data[stat_data['hand'] == 'Left']['rt_ms']
    
    if len(stat_right) > 0 and len(stat_left) > 0:
        bins = np.linspace(
            min(stat_right.min(), stat_left.min()),
            max(stat_right.max(), stat_left.max()),
            30
        )
        
        axes[0].hist(stat_right, bins=bins, alpha=0.6, label='Right', color='blue', edgecolor='black')
        axes[0].hist(stat_left, bins=bins, alpha=0.6, label='Left', color='red', edgecolor='black')
        axes[0].set_xlabel('Reaction Time (ms)', fontsize=12)
        axes[0].set_ylabel('Count', fontsize=12)
        axes[0].set_title('STAT Blocks: RT Distribution by Hand', fontsize=14, fontweight='bold')
        axes[0].legend(fontsize=11)
        axes[0].grid(alpha=0.3)
    
    # Combined
    axes[1].hist(stat_data['rt_ms'], bins=30, alpha=0.7, color='green', edgecolor='black')
    axes[1].set_xlabel('Reaction Time (ms)', fontsize=12)
    axes[1].set_ylabel('Count', fontsize=12)
    axes[1].set_title('STAT Blocks: Combined RT Distribution', fontsize=14, fontweight='bold')
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('kinarm_rt_dist_stat.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ STAT figure saved as 'kinarm_rt_dist_stat.png'")

In [ ]:
# INTER blocks
inter_data = trial_data[trial_data['block_type'] == 'INTER']

if len(inter_data) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    conditions = [
        ('Right', 75, 'Right 75°/s', 'blue'),
        ('Left', 75, 'Left 75°/s', 'red'),
        ('Right', 150, 'Right 150°/s', 'darkblue'),
        ('Left', 150, 'Left 150°/s', 'darkred')
    ]
    
    for idx, (hand, speed, label, color) in enumerate(conditions):
        ax = axes[idx // 2, idx % 2]
        data = inter_data[(inter_data['hand'] == hand) & (inter_data['speed'] == speed)]['rt_ms']
        
        if len(data) > 0:
            ax.hist(data, bins=20, alpha=0.7, color=color, edgecolor='black')
            ax.set_xlabel('Reaction Time (ms)', fontsize=11)
            ax.set_ylabel('Count', fontsize=11)
            ax.set_title(f'INTER: {label}', fontsize=13, fontweight='bold')
            ax.grid(alpha=0.3)
            
            # Add statistics box
            stats_text = f'n={len(data)}\nμ={data.mean():.1f}ms\nσ={data.std():.1f}ms'
            ax.text(0.95, 0.95, stats_text,
                   transform=ax.transAxes, ha='right', va='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7),
                   fontsize=10)
    
    plt.tight_layout()
    plt.savefig('kinarm_rt_dist_inter.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ INTER figure saved as 'kinarm_rt_dist_inter.png'")

## 6. Fit Drift Diffusion Models

In [ ]:
if not PYDDM_AVAILABLE:
    print("⚠ PyDDM is not installed. Install it to run DDM fitting.")
    print("  pip install pyddm --break-system-packages")
else:
    print("Starting DDM fitting...\n")

In [ ]:
def prepare_sample(data, condition_filter):
    """Prepare PyDDM Sample from filtered data."""
    filtered = data.copy()
    for col, val in condition_filter.items():
        filtered = filtered[filtered[col] == val]
    
    rts = filtered['rt_seconds'].values
    choices = np.ones(len(rts))  # All correct
    
    sample = Sample.from_numpy_array(rts, choices)
    return sample, len(rts)


def fit_ddm(sample, label):
    """Fit a 3-parameter DDM."""
    model = Model(
        name=label,
        drift=DriftConstant(drift=Fittable(minval=-10, maxval=20)),
        noise=NoiseConstant(noise=1),
        bound=BoundConstant(B=Fittable(minval=0.2, maxval=3)),
        IC=ICPointSourceCenter(),
        overlay=OverlayNonDecision(nondectime=Fittable(minval=0.1, maxval=1))
    )
    
    fitted_model = fit_adjust_model(sample=sample, model=model, verbose=False)
    return fitted_model


def extract_parameters(model):
    """Extract parameters from fitted model."""
    params = {}
    
    drift_param = model.get_dependence('drift').drift
    params['drift'] = float(drift_param.real() if hasattr(drift_param, 'real') else drift_param)
    
    bound_param = model.get_dependence('bound').B
    params['bound'] = float(bound_param.real() if hasattr(bound_param, 'real') else bound_param)
    
    ndt_param = model.get_dependence('overlay').nondectime
    params['nondectime'] = float(ndt_param.real() if hasattr(ndt_param, 'real') else ndt_param)
    
    return params

if PYDDM_AVAILABLE:
    print("✓ DDM functions defined")

In [ ]:
if PYDDM_AVAILABLE:
    # Define conditions to fit
    conditions = []
    
    # STAT - pooled
    stat_data = trial_data[trial_data['block_type'] == 'STAT']
    if len(stat_data) > 0:
        conditions.append({
            'filter': {'block_type': 'STAT'},
            'label': 'STAT_pooled',
            'description': 'Stationary (both hands)'
        })
    
    # INTER by speed
    inter_speeds = trial_data[trial_data['block_type'] == 'INTER']['speed'].unique()
    for speed in sorted([s for s in inter_speeds if pd.notna(s)]):
        conditions.append({
            'filter': {'block_type': 'INTER', 'speed': speed},
            'label': f'INTER_{int(speed)}deg',
            'description': f'Interception {int(speed)}°/s (both hands)'
        })
    
    print(f"Will fit {len(conditions)} conditions\n")

In [ ]:
if PYDDM_AVAILABLE:
    models = {}
    results = []
    
    for cond in conditions:
        try:
            sample, n_trials = prepare_sample(trial_data, cond['filter'])
            
            if n_trials < 10:
                print(f"⚠ Skipping {cond['label']}: insufficient trials (n={n_trials})")
                continue
            
            print(f"Fitting {cond['description']} (n={n_trials})...")
            model = fit_ddm(sample, cond['label'])
            models[cond['label']] = model
            
            params = extract_parameters(model)
            result = {
                'Condition': cond['description'],
                'n_trials': n_trials,
                'drift': params['drift'],
                'bound': params['bound'],
                'nondectime': params['nondectime']
            }
            results.append(result)
            
            print(f"  ✓ drift={params['drift']:.3f}, "
                  f"bound={params['bound']:.3f}, "
                  f"nondectime={params['nondectime']:.3f}\n")
        
        except Exception as e:
            print(f"⚠ Error fitting {cond['label']}: {e}\n")
    
    # Create results dataframe
    if len(results) > 0:
        results_df = pd.DataFrame(results)
        print("\n" + "="*60)
        print("DDM RESULTS SUMMARY")
        print("="*60)
        display(results_df)
        
        # Save results
        results_df.to_csv('kinarm_ddm_results.csv', index=False)
        print("\n✓ Results saved to 'kinarm_ddm_results.csv'")

## 7. Visualize DDM Parameters

In [ ]:
if PYDDM_AVAILABLE and len(results) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    params = ['drift', 'bound', 'nondectime']
    titles = ['Drift Rate (v)', 'Boundary (a)', 'Non-Decision Time (t₀)']
    colors = ['steelblue', 'coral', 'mediumseagreen']
    
    for idx, (param, title, color) in enumerate(zip(params, titles, colors)):
        ax = axes[idx]
        ax.bar(range(len(results_df)), results_df[param], color=color, alpha=0.7, edgecolor='black')
        ax.set_xticks(range(len(results_df)))
        ax.set_xticklabels(results_df['Condition'], rotation=45, ha='right')
        ax.set_ylabel(title, fontsize=12)
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.grid(alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('kinarm_ddm_parameters.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Parameter plot saved as 'kinarm_ddm_parameters.png'")

## 8. Summary

The analysis is complete! You should now have:

1. **kinarm_trial_summary.csv** - All trials with metadata and RTs
2. **kinarm_rt_dist_stat.png** - RT distributions for STAT blocks
3. **kinarm_rt_dist_inter.png** - RT distributions for INTER blocks
4. **kinarm_ddm_results.csv** - Fitted DDM parameters
5. **kinarm_ddm_parameters.png** - Visualization of DDM parameters

### Interpreting DDM Parameters

- **Drift rate (v)**: Speed of evidence accumulation (higher = faster decisions)
- **Boundary (a)**: Decision threshold (higher = more cautious, slower but more accurate)
- **Non-decision time (t₀)**: Sensory/motor delays (time not spent deciding)